# broharness recipe

This notebook shows how to *use* `broharness` as a packaged library --
contrast with `from_scratch.ipynb`, which hand-assembles `SkillControl`,
`ToolControl`, the `TaskRegistry`, and all six flow tasks cell by cell.
`Harness` packages exactly that fixed wiring (skill_call -> tool_call ->
tool_use -> answer, with ask_user_question/fail_recovery as shared
side-routes), built once instead of by hand per notebook/script.

`Harness` is deliberately lean: it owns nothing about a run except that
fixed orchestration. `State` -- root, skill_dir, skill_control,
tool_control, tools, session_tools, system_prompt, model_id, verbose,
everything -- is built entirely outside `Harness`, the same way
`from_scratch.ipynb` builds it. This keeps the two testable and
controllable separately: swap what's in `State` without touching
`Harness`, or reuse one `Harness` across many independently-built `State`s.

```python
from broharness import Harness
from broharness.data_model import State

h = Harness()
state = State(...)     # built directly -- see from_scratch.ipynb
state = h.run(state)   # Harness just runs the fixed flow over it
```</cell id="68f84879">


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from broskill import SkillControl, ToolControl

from broharness import Harness
from broharness.data_model import State, LLMUse
from broharness.llms.bedrock import UserMessage
from broharness.debug import print_debug

ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

# Same plumbing as from_scratch.ipynb -- SkillControl/ToolControl aren't
# Harness's concern, they're State's. Built once here and reused across
# every State below, just like the notebook this is modeled on.
sc = SkillControl(SKILL_DIR)
tc = ToolControl(sc)
TOOLS = dict(
    load_skill=sc.load_skill,
    load_skill_extension=sc.load_skill_extension,
    load_tool=tc.load_tool,
)

system_prompt = "You're Andy who is the best bro in the world. Always respond in bro-tone with chill and mellow manner."

# Harness itself takes no config -- it's the same fixed flow regardless of
# which State it's handed, so one instance is reused for every example below.
h = Harness()

## 1. One-shot request

Build `State` directly (exactly like `from_scratch.ipynb`'s test cells),
then hand it to `h.run()`.

In [3]:
# content = "what folders are directly under skills/?"
content = "I wanna understand how A/B Testing works."
messages = [UserMessage(content)]
state = State(
    root=ROOT,
    skill_dir=SKILL_DIR,
    messages=messages,
    session_messages=messages.copy(),
    system_prompt=system_prompt,
    skill_control=sc,
    tool_control=tc,
    tools=TOOLS,
    session_tools=TOOLS.copy(),
    debug=messages.copy(),
    verbose=True,   # mirrors from_scratch.ipynb's unconditional print(__file__) traces
)
state

State(root=WindowsPath('D:/study-on-agent'), skill_dir=WindowsPath('D:/study-on-agent/skills'), messages=[{'role': 'user', 'content': [{'text': 'I wanna understand how A/B Testing works.'}]}], session_messages=[{'role': 'user', 'content': [{'text': 'I wanna understand how A/B Testing works.'}]}], skill_control=<broskill.processing.skill.SkillControl object at 0x0000021770D61E50>, tool_control=<broskill.processing.tool.ToolControl object at 0x00000217714F1100>, tools={'load_skill': <bound method SkillControl.load_skill of <broskill.processing.skill.SkillControl object at 0x0000021770D61E50>>, 'load_skill_extension': <bound method SkillControl.load_skill_extension of <broskill.processing.skill.SkillControl object at 0x0000021770D61E50>>, 'load_tool': <bound method ToolControl.load_tool of <broskill.processing.tool.ToolControl object at 0x00000217714F1100>>}, session_tools={'load_skill': <bound method SkillControl.load_skill of <broskill.processing.skill.SkillControl object at 0x000002177

In [4]:
state = h.run(state)
print(state.messages[-1]['content'][0]['text'])

D:\study-on-agent\src\broharness\flows\skill_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill', 'input': {'skill_name': 'ds-mentor'}}]
load_skill passed
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill_extension', 'input': {'skill_name': 'ds-mentor', 'path': 'references/stats-basics.md'}}]
load_skill_extension passed
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill_extension', 'input': {'skill_name': 'ds-mentor', 'path': 'references/metrics.md'}}]
load_skill_extension passed
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill_extension', 'input': {'skill_name': 'stats-basics', 'path': 'references/stats-basics.md'}}]
Skill 'stats-basics' hasn't been loaded in this conversation -- call load_skill with skill_name='stats-basics' first, o

In [25]:
state.messages

[{'role': 'user',
  'content': [{'text': 'I wanna understand how A/B Testing works.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Alright bro, let\'s break down A/B testing. It\'s all about figuring out if one thing is *actually* better than another, not just *seemingly* so.\n\nEssentially, A/B testing comes down to hypothesis testing—that\'s the core concept from the `stats-basics` reference. You\'ve got your null hypothesis, which says "there\'s no real difference between version A and version B." Then you have your alternative hypothesis, which says "there *is* a real difference."\n\nThe goal of the A/B test is to gather data and use it to see how likely it is that you\'d see the results you *did* see if the null hypothesis was true.  A small p-value (typically below 0.05) suggests that the observed difference is unlikely due to random chance, so you\'d reject the null hypothesis and conclude that one version *is* better than the other.\n\nThink of it like this: Let\'s say you

In [5]:
state.debug

[{'role': 'user',
  'content': [{'text': 'I wanna understand how A/B Testing works.'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill", "input": { "skill_name": "ds-mentor" } }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 1187, 'outputTokens': 47, 'totalTokens': 1234}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill_extension", "input": { "skill_name": "ds-mentor", "path": "references/stats-basics.md" } }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 1672, 'outputTokens': 61, 'totalTokens': 1733}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill_extension", "input": { "skill_name": "ds-mentor", "path": "references/metrics.md" } }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 2284, 'outputTokens': 59, 'totalTokens': 2343}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill_exte

## 2. Inspecting what happened

Same `State` fields as `from_scratch.ipynb` -- `Harness` doesn't hide anything, it just builds the plumbing for you.

In [13]:
print_debug(state.debug)

[1] user: [free text] what folders are directly under skills/?
[2] assistant: load_skill({'skill_name': 'file-ops'})  (in=1068 out=47)
[3] assistant: list_directory({'pattern': 'skills/*'})  (in=3869 out=54)
[4] assistant: list_directory({'pattern': 'skills/*'})  (in=3909 out=54)
[5] assistant: [free text] Hey bro, chillin'.

Looks like the folders directly under "skills/" are: `file-ops/`, `skill-call/`, `tell-joke/`, and `tool-call/`. Just wanted to let you know what's hangin' around in there, you know?  (in=2466 out=63)
[6] assistant: [free text] Alright, cool, bro. Let's peep inside that `tell-joke/` folder.

```tool_code
tool_code:
```
```tool_code
scripts/list_directory.py tell-joke/*
```  (in=2553 out=52)


In [14]:
state.tool_results

[{'list_directory': 'skills\\file-ops/\nskills\\skill-call/\nskills\\tell-joke/\nskills\\tool-call/\n'}]

In [15]:
state.usage

{'SKILL_CALL': {'google.gemma-3-12b-it': {'input_tokens': 1068,
   'output_tokens': 47,
   'call_count': 1}},
 'TOOL_CALL': {'google.gemma-3-12b-it': {'input_tokens': 7778,
   'output_tokens': 108,
   'call_count': 2}},
 'ANSWER': {'google.gemma-3-12b-it': {'input_tokens': 5019,
   'output_tokens': 115,
   'call_count': 2}}}

In [16]:
def cal_usage(input, output, input_price, output_price, currency=1, session=1):
    mil = 1_000_000
    inputPrice = (input/mil)*input_price*currency*session
    outputPrice = (output/mil)*output_price*currency*session
    return inputPrice, outputPrice, inputPrice+outputPrice

for slot, models in state.usage.items():
    for model_id, u in models.items():
        i, o, t = cal_usage(u['input_tokens'], u['output_tokens'], 0.09, 0.29)
        print(f"{slot:12s} {model_id}: ${t:.5f}  (in={u['input_tokens']} out={u['output_tokens']} calls={u['call_count']})")

SKILL_CALL   google.gemma-3-12b-it: $0.00011  (in=1068 out=47 calls=1)
TOOL_CALL    google.gemma-3-12b-it: $0.00073  (in=7778 out=108 calls=2)
ANSWER       google.gemma-3-12b-it: $0.00049  (in=5019 out=115 calls=2)


## 3. Continuing a conversation (multi-turn)

`run()` doesn't touch messages or turn bookkeeping at all -- that's on the
caller now. To continue a conversation with the same loaded skills/tools:

1. `state.flush_turn()` -- resets everything that's only valid for the turn
   that just finished (`tool_results`, `candidated_tools`, `executed_calls`,
   `error_message`, `return_to`, `retry_count`), while keeping
   `registered_skills`, `registered_tools`, `session_tools`, and the full
   `messages`/`debug`/`usage` history intact.
2. Append the next request as a `UserMessage` to both `state.messages` and
   `state.session_messages`.
3. `h.run(state)` again.

Skipping step 1 is exactly the bug that caused a real hallucination earlier
in this project: a previous turn's `tool_results` was still sitting in
`State` when `Answer` built the next turn's prompt, so it answered using
stale content instead of the new question's actual result.

In [ ]:
from broharness.llms.bedrock import UserMessage

next_request = "what about just skill-call?"
state.flush_turn()
state.messages.append(UserMessage(next_request))
state.session_messages.append(UserMessage(next_request))
state = h.run(state)
print(state.messages[-1]['content'][0]['text'])

In [ ]:
print("total turns in this conversation:", len(state.messages) // 2)
for m in state.messages:
    print(f"{m['role']:9s} {m['content'][0]['text']}")

## 4. file-ops: a real side effect (create_file)

`create_file` doesn't require confirmation (only `update_file`/`delete_file`
do, per `skills/file-ops/SKILL.md` -- see the note below). A fresh `State`
for this, since it's a new, unrelated request -- same `h`, since `Harness`
carries no per-conversation config to isolate.

In [ ]:
messages2 = [UserMessage("create a file called scratch/recipe-demo.md with the content 'created from recipe.ipynb'")]
state2 = State(
    root=ROOT,
    skill_dir=SKILL_DIR,
    messages=messages2,
    session_messages=messages2.copy(),
    skill_control=sc,
    tool_control=tc,
    tools=TOOLS,
    session_tools=TOOLS.copy(),
    debug=messages2.copy(),
    verbose=True,
)
state2 = h.run(state2)
print(state2.messages[-1]['content'][0]['text'])

In [ ]:
(ROOT / 'scratch' / 'recipe-demo.md').read_text()

## 5. Destructive ops ask first -- not auto-run here

`update_file` and `delete_file` are gated behind `ask_user_question` in
`skills/file-ops/SKILL.md`'s instructions, which calls Python's real
`input()` (see `flows/ask_user_question.py`) -- that blocks waiting for a
reply, so it isn't run automatically in this cell the way the cells above
are. Uncomment and run this yourself interactively to see the confirmation
step actually pause and wait for your answer:

In [ ]:
# next_request = "delete scratch/recipe-demo.md"
# state2.flush_turn()
# state2.messages.append(UserMessage(next_request))
# state2.session_messages.append(UserMessage(next_request))
# state2 = h.run(state2)
# print(state2.messages[-1]['content'][0]['text'])

## 6. Quiet mode

`verbose` lives on `State`, not `Harness` -- same `h`, a `State` built with
`verbose=False` instead. No per-task trace prints, only the final answer.
`state.debug` still has the full call log either way; `verbose` only
controls what prints to stdout as it runs.

In [ ]:
messages3 = [UserMessage("what folders are directly under skills/?")]
state3 = State(
    root=ROOT,
    skill_dir=SKILL_DIR,
    messages=messages3,
    session_messages=messages3.copy(),
    skill_control=sc,
    tool_control=tc,
    tools=TOOLS,
    session_tools=TOOLS.copy(),
    debug=messages3.copy(),
    verbose=False,
)
state3 = h.run(state3)
print(state3.messages[-1]['content'][0]['text'])